 # 🏆 RadioStripe Deployment Competition

 **Goal:** Optimize a sub-THz RadioStripe deployment to **maximize uplink
 coverage** across a room full of users.

 You just completed the tutorial and learned how the simulator works end-to-end.
 Now it's time to put that knowledge to work! Your task is to design the best
 possible RadioStripe deployment for a fixed room and a fixed set of UE
 positions.

 ## Rules

 1. **Fixed room** — the room dimensions are given and **cannot be changed**.
 2. **Fixed waveform** — modulation (QPSK), bandwidth (3 GHz), and number of
    OFDM symbols are fixed and **cannot be changed**.
 3. **Fixed UE positions** — you optimize on a *training set* of UEs (provided
    below). The final ranking will be evaluated on a **hidden test set** with a
    different random seed.
 4. **RU budget** — you may use **at most `MAX_TOTAL_RUS` Radio Units** in
    total across all stripes.
 5. **Fiber length must match RU spacing** — the `fiber.length` in the
    component config must equal the physical spacing between adjacent RUs on
    each stripe. No cheating with zero-length fibers!

 ## What You Can Optimize

 | Knob | How | Trade-off |
 |---|---|---|
 | **Stripe / RU geometry** | Change `n_stripes`, `n_rus_per_stripe`, `ru_spacing_m`, `stripe_spacing_m` | Coverage area vs. component count |
 | **RU spacing** | Adjust `ru_spacing_m` | Shorter fiber → less PA saturation, but fewer RUs cover less area |
 | **TX power** | Change `TX_POWER_DBM` in the waveform config | Higher power → better SNR but more PA distortion |
 | **Beam steering** | Set `UE_BEAM` and `RU_BEAM` per UE | Directivity gain vs. alignment risk |
 | **Active RU selection** | Implement `custom_select_active_ru()` | Distance isn't always optimal — fewer booster hops = less distortion |
 | **Creative ideas** | Anything else within the rules! | Surprise us! |

 ## Scoring

 Your deployment is evaluated across **all UE positions**. For each UE the
 full uplink chain is simulated and the EVM (Error Vector Magnitude) is
 recorded. Two leaderboards rank participants:

 ### Leaderboard 1 — Signal Quality Score
 $$\text{Score}_1 = \text{mean}(\text{EVM}) + \lambda \cdot \max(\text{EVM})$$
 where $\lambda = 0.5$. This penalizes both poor average quality **and**
 leaving any single UE with terrible reception. **Lower is better.**

 ### Leaderboard 2 — Cost-Aware Score
 $$\text{Score}_2 = \text{mean}(\text{EVM}) + \lambda \cdot \max(\text{EVM}) + \mu \cdot N_{\text{RUs}}$$
 where $\lambda = 0.5$ and $\mu = 0.5$. This additionally penalizes using
 more hardware — rewarding efficient deployments. **Lower is better.**

 ## How to Submit

 1. Edit **only** the `# === YOUR CODE HERE ===` sections below.
 2. Run the full script to see your scores.
 3. Report your two scores and your team name.

 Good luck!

In [ ]:
# clone simulator repo
!rm -rf /content6GTandem-simulator
!git clone https://github.com/6GTandem/6GTandem-simulator.git

In [ ]:
# ===========================================================================
# Boilerplate — DO NOT MODIFY
# ===========================================================================
import os
import sys
from copy import deepcopy
from pathlib import Path
import copy

from google.colab import auth
import gspread
from google.auth import default

import matplotlib.pyplot as plt
import numpy as np
import yaml
from scipy.signal import welch

# Resolve repository root so this notebook can run from any working directory.
root = Path('/content/6GTandem-simulator')
while not (root / "wireless_channel").exists() and root != root.parent:
    root = root.parent

if not (root / "wireless_channel").exists():
    raise RuntimeError("Could not locate repository root containing 'wireless_channel'.")

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print(f"Repository root: {root}")

In [ ]:
# install dependencies
!pip install -q -r requirements.txt
# if trouble:
# !pip install -r requirements.txt

In [ ]:
# ===========================================================================
# Boilerplate — DO NOT MODIFY
# ===========================================================================
from examples.training_school.utils import (
    build_radio_stripe_config,
    generate_grid_ue_positions,
    generate_random_ue_positions,
    select_active_radio_unit,
)
from plotter import plotter
from sub_THz_stripe.amplifier.amplifier import Amplifier
from sub_THz_stripe.central_unit.central_unit import CentralUnit
from sub_THz_stripe.combiner.combiner import Combiner
from sub_THz_stripe.coupler.coupler import Coupler
from sub_THz_stripe.phase_shifter.phase_shifter import PhaseShifter
from sub_THz_stripe.radio_unit.radio_unit import RadioUnit
from sub_THz_stripe.radiostripe.radiostripe import RadioStripe
from sub_THz_stripe.splitter.splitter import Splitter
from wireless_channel.subTHz_channel import build_channel
from wireless_channel.waveforms import Waveform


In [ ]:
# ===========================================================================
# FIXED PARAMETERS — DO NOT MODIFY
# ===========================================================================

# --- Room (fixed) -----------------------------------------------------------
ROOM_X = 10.0   # metres
ROOM_Y = 20.0   # metres
ROOM_Z = 3.5    # metres

# --- RU budget (fixed) ------------------------------------------------------
MAX_TOTAL_RUS = 15  # maximum number of Radio Units across all stripes

# --- Scoring weights (fixed) ------------------------------------------------
LAMBDA = 0.5   # weight for max(EVM)
MU = 0.5       # weight for N_total_RUs (leaderboard 2 only)

# --- Training UE set (fixed) ------------------------------------------------
N_UES_TRAINING = 50
UE_Z = 1.5
UE_WALL_MARGIN = 0.1
UE_SEED_TRAINING = 42   # known seed — you can optimize on this

# --- Grid UE set (fixed) ---------------------------------------------------
N_GRID_X = 5             # grid points along X
N_GRID_Y = 10            # grid points along Y

# --- Waveform (fixed) -------------------------------------------------------
configs_dir = root / "examples" / "training_school" / "configs"
with open(configs_dir / "tutorial_waveform_config.yaml", "r", encoding="utf8") as _f:
    WAVEFORM_CONFIG_BASE = yaml.safe_load(_f)

 ## ✏️ Your Solution

 Edit the sections below. You may change anything inside the
 `# === YOUR CODE HERE ===` blocks. Do **not** change the fixed parameters
 or the evaluation loop.

In [ ]:
# ===========================================================================
# SECTION A — Deployment geometry
# ===========================================================================
# Choose how many stripes and RUs to use (within the budget), their spacing,
# and any other geometry parameters.
#
# Constraint: N_STRIPES * N_RUS_PER_STRIPE <= MAX_TOTAL_RUS
#
# === YOUR CODE HERE === (modify the values below) ==========================

TEAM_NAME = "Team Default" # todo set your team name!

N_STRIPES = 5
N_RUS_PER_STRIPE = 3          # 5 * 3 = 15 RUs total (max budget)
FIRST_STRIPE_X = 1.0             # x-coordinate of the first stripe 
FIRST_STRIPE_Y = 1.0             # y-coordinate of the first stripe 
RU_SPACING_M = 3.0            # metres between adjacent RUs
STRIPE_SPACING_M = 1.5        # metres between stripes
TX_POWER_DBM = 30             # transmit power in dBm

# === END YOUR CODE ===========================================================

# --- Validate RU budget -----------------------------------------------------
N_TOTAL_RUS = N_STRIPES * N_RUS_PER_STRIPE
assert N_TOTAL_RUS <= MAX_TOTAL_RUS, (
    f"Budget exceeded! {N_TOTAL_RUS} RUs used, but maximum is {MAX_TOTAL_RUS}."
)

# TODO ALLOW FOR RANDOM CONFIGS? NOT JUST RECTANGULAR


In [ ]:
# %%
# ===========================================================================
# SECTION B — Active RU selection strategy
# ===========================================================================
# The default uses distance-based selection (closest RU to the UE).
# You may implement a smarter strategy here.
#
# Your function receives:
#   - ue_pos : dict with keys "x", "y", "z"
#   - radio_stripes : the stripe config (list of lists)
#   - component_config : the component config dict
#
# It must return:
#   - (stripe_idx, ru_idx)
#
# === YOUR CODE HERE === (edit the function body) =============================

RU_SELECTION_MODE = "random"

def custom_select_active_ru(ue_pos, radio_stripes, component_config):
    """Select the active RU for a given UE position.

    Default: pick the closest RU by Euclidean distance.
    You can replace this with any strategy you like.
    """
    stripe_idx, ru_idx, _, _ = select_active_radio_unit(
        ue_position=ue_pos,
        radio_stripes=radio_stripes,
        mode=RU_SELECTION_MODE,
    )
    return stripe_idx, ru_idx

# === END YOUR CODE ===========================================================

In [ ]:
# %%
# ===========================================================================
# SECTION C — Beam steering strategy
# ===========================================================================
# Return (ue_beam, ru_beam) for each UE. Default: broadside (0, 0).
#
# === YOUR CODE HERE === (edit the function body) =============================

BEAM_MODE = "random"

def get_beam_indices(ue_pos, active_ru_pos):
    """Return (ue_beam_angle, ru_beam_angle) for a given UE-RU pair.

    Modes:
    - "zero":   broadside beam (0° steering) for both UE and RU.
    - "random": random beam angles in [-90, 90] degrees for both.
    - "custom": user-defined logic (edit the custom branch below).
    """
    if BEAM_MODE == "zero":
        return 0, 0
    elif BEAM_MODE == "random":
        ue_beam = np.random.uniform(-90, 90)
        ru_beam = np.random.uniform(-90, 90)
        return ue_beam, ru_beam
    elif BEAM_MODE == "custom":
        # === YOUR CUSTOM BEAM LOGIC HERE ===
        raise NotImplementedError("Implement your custom beam steering logic!")
    else:
        raise ValueError(f"Unknown BEAM_MODE: '{BEAM_MODE}'. Use 'zero', 'random', or 'custom'.")

# === END YOUR CODE ===========================================================

 ## Evaluation Loop — DO NOT MODIFY

 The code below builds your deployment, loops over every UE position,
 simulates the full uplink chain, and computes the two scores.

In [ ]:
# ===========================================================================
# BUILD DEPLOYMENT
# ===========================================================================

# --- Waveform config with your TX power ------------------------------------
waveform_config = copy.deepcopy(WAVEFORM_CONFIG_BASE)
waveform_config["tx_power"] = TX_POWER_DBM

# --- Environment config from your geometry ----------------------------------
env_config = build_radio_stripe_config(
    room_x=ROOM_X,
    room_y=ROOM_Y,
    room_z=ROOM_Z,
    n_stripes=N_STRIPES,
    n_rus_per_stripe=N_RUS_PER_STRIPE,
    ru_spacing_m=RU_SPACING_M,
    stripe_spacing_m=STRIPE_SPACING_M,
    x_first_stripe=FIRST_STRIPE_X,
    y_margin=FIRST_STRIPE_Y
)

# --- UE positions (training set) -------------------------------------------
ue_positions_random = generate_random_ue_positions(
    n_users=N_UES_TRAINING,
    room_x=ROOM_X,
    room_y=ROOM_Y,
    z_height=UE_Z,
    wall_margin=UE_WALL_MARGIN,
    seed=UE_SEED_TRAINING,
)

# --- UE positions (grid — uniformly sampled) --------------------------------
ue_positions_grid = generate_grid_ue_positions(
    n_x=N_GRID_X,
    n_y=N_GRID_Y,
    room_x=ROOM_X,
    room_y=ROOM_Y,
    z_height=UE_Z,
    wall_margin=UE_WALL_MARGIN,
)

N_RANDOM_UES = len(ue_positions_random)
ue_positions = ue_positions_random + ue_positions_grid
env_config["ue_positions"] = ue_positions

# --- Component config (fiber length must match RU spacing) ------------------
with open(configs_dir / "tutorial_component_config.yaml", "r", encoding="utf8") as _f:
    component_config = yaml.safe_load(_f)
component_config["fiber"]["length"] = float(RU_SPACING_M)

# --- Build waveform and generate TX signal once ----------------------------
freq_band_config = env_config["sub_thz"]
wf = Waveform.from_config(waveform_config, freq_band_config)

bits = wf.generate_bits()
qam = wf.qam_modulate()
ofdm_time = wf.ofdm_modulate()
x_combined_freq = wf.ofdm_time_to_freq(ofdm_time)
xsubc = wf.extract_subcarriers(x_combined_freq)

# --- Build UE transmit chain (fixed) ---------------------------------------
ue = CentralUnit()
ofdm_time_after_ue = ue.run(ofdm_time)

amp = Amplifier(bw=env_config["sub_thz"]["bw"])
coup = Coupler(wf=wf)
split = Splitter(env_config["antenna"]["N_antennas"])
comb = Combiner()
ps = PhaseShifter(
    num_shifters=env_config["antenna"]["N_antennas"],
    resolution=int(component_config["phase_shifter"]["resolution"]),
)
ue_ru = RadioUnit(
    x=0, y=0, z=0,
    boost_amp=amp, antenna_amp=amp,
    coup_in=coup, coup_out=coup,
    splitter=split, combiner=comb, pshift=ps,
)

# --- Build all stripes from your config ------------------------------------
stripes: list[RadioStripe] = []
for stripe_cfg in env_config["radio_stripes"]:
    stripes.append(
        RadioStripe.from_config_locations(
            stripe_cfg, component_config,
            env_config["antenna"]["N_antennas"], wf,
        )
    )

# --- Plot the room layout ---------------------------------------------------
plotter.plot_room(env_config)
plt.title(f"{TEAM_NAME} — Deployment ({N_TOTAL_RUS} RUs)")
plt.show()

print(f"\n{'=' * 60}")
print(f"  Team:     {TEAM_NAME}")
print(f"  Stripes:  {N_STRIPES}")
print(f"  RUs/stripe: {N_RUS_PER_STRIPE}  (total: {N_TOTAL_RUS})")
print(f"  RU spacing: {RU_SPACING_M} m")
print(f"  TX power:   {TX_POWER_DBM} dBm")
print(f"  UEs:        {len(ue_positions)}")
print(f"{'=' * 60}")

In [ ]:
# ===========================================================================
# EVALUATE ACROSS ALL UEs
# ===========================================================================

evm_per_ue = []
ber_per_ue = []

for ue_idx, ue_pos in enumerate(ue_positions):
    # --- Active RU selection (your function) --------------------------------
    stripe_idx, ru_idx = custom_select_active_ru(
        ue_pos, env_config["radio_stripes"], component_config,
    )

    # --- Beam steering (your function) --------------------------------------
    ru_entry = env_config["radio_stripes"][stripe_idx][ru_idx + 1]["radio_unit"]
    ue_beam, ru_beam = get_beam_indices(ue_pos, ru_entry)

    # --- UE transmit --------------------------------------------------------
    ue_shift = ue_ru.phase_shifter.get_phases(ue_beam)
    iq_data_tx, _ = ue_ru.transmit(ofdm_time_after_ue, ue_shift)

    # --- Wireless channel ---------------------------------------------------
    channel = build_channel(
        channel_model="los",
        ue_coordinates=ue_pos,
        component_config=component_config,
        stripe_positions=env_config["radio_stripes"],
        waveform=wf,
        Nr_ue_antennas=env_config["antenna"]["N_antennas"],
        Nr_ru_antennas=env_config["antenna"]["N_antennas"],
        debug=False,
        los_normalize_gain=False,
    )
    iq_data_rx = channel.transmit_ul_id(iq_data_tx, stripe_idx, ru_idx, wf)

    # --- Stripe receive -----------------------------------------------------
    stripe = stripes[stripe_idx]
    stripe.active_unit = ru_idx
    ru_shift = stripe.radio_units[0].phase_shifter.get_phases(ru_beam)
    y, _ = stripe.receive(iq_data_rx, ru_shift)

    # --- OFDM demodulation + equalization -----------------------------------
    y_combined_freq = wf.ofdm_time_to_freq(y)
    rx_subc = wf.extract_subcarriers(y_combined_freq)
    H_est = wf.channel_estimate_ls(rx_subc, interp_mode="phase")
    eq_subc = wf.equalize_one_tap(rx_subc, H_est)

    # --- Metrics ------------------------------------------------------------
    evm = wf.compute_evm(xsubc, eq_subc)
    y_qam = wf.demap_data_from_grid(eq_subc).flatten()
    y_bits = wf.qam_to_bits(y_qam)
    ber_val = wf.compute_ber(bits, y_bits)

    evm_per_ue.append(evm)
    ber_per_ue.append(ber_val)

    print(f"  UE {ue_idx:2d}  stripe={stripe_idx} ru={ru_idx:2d}  "
          f"EVM={evm:7.2f}%  BER={ber_val:.3e}")

evm_arr = np.array(evm_per_ue)
ber_arr = np.array(ber_per_ue)

In [ ]:
# ===========================================================================
# COMPUTE SCORES
# ===========================================================================

mean_evm = np.mean(evm_arr)
max_evm = np.max(evm_arr)
score_1 = mean_evm + LAMBDA * max_evm
score_2 = mean_evm + LAMBDA * max_evm + MU * N_TOTAL_RUS

print(f"\n{'=' * 60}")
print(f"  RESULTS — {TEAM_NAME}")
print(f"{'=' * 60}")
print(f"  UEs evaluated     : {len(evm_arr)}")
print(f"  Mean EVM           : {mean_evm:.2f} %")
print(f"  Max  EVM           : {max_evm:.2f} %")
print(f"  Min  EVM           : {np.min(evm_arr):.2f} %")
print(f"  Std  EVM           : {np.std(evm_arr):.2f} %")
print(f"  Mean BER           : {np.mean(ber_arr):.3e}")
print(f"  Total RUs          : {N_TOTAL_RUS}")
print(f"{'─' * 60}")
print(f"  Leaderboard 1 (quality):    {score_1:.4f}")
print(f"    = mean({mean_evm:.2f}) + {LAMBDA}·max({max_evm:.2f})")
print(f"  Leaderboard 2 (cost-aware): {score_2:.4f}")
print(f"    = mean({mean_evm:.2f}) + {LAMBDA}·max({max_evm:.2f}) + {MU}·{N_TOTAL_RUS}")
print(f"{'=' * 60}")

In [ ]:
# ===========================================================================
# VISUALIZATION
# ===========================================================================

# --- EVM per UE bar chart ---------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 4))
colors = plt.cm.RdYlGn_r(evm_arr / max(evm_arr.max(), 1))  # red=bad, green=good
ax.bar(range(len(evm_arr)), evm_arr, color=colors, edgecolor="gray", linewidth=0.5)
ax.axhline(mean_evm, color="blue", linestyle="--", linewidth=1.2, label=f"Mean EVM = {mean_evm:.2f}%")
ax.axhline(max_evm, color="red", linestyle="--", linewidth=1.2, label=f"Max EVM = {max_evm:.2f}%")
ax.set_xlabel("UE index")
ax.set_ylabel("EVM (%)")
ax.set_title(f"{TEAM_NAME} — EVM per UE  |  Score₁ = {score_1:.2f}  |  Score₂ = {score_2:.2f}")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- Room plot coloured by EVM per UE ---------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))

# Plot stripes and RUs
for s_idx, stripe_cfg in enumerate(env_config["radio_stripes"]):
    cu = stripe_cfg[0]["central_unit"]
    ax.plot(cu["x"], cu["y"], "ks", markersize=8)
    for ru_entry in stripe_cfg[1:]:
        ru = ru_entry["radio_unit"]
        ax.plot(ru["x"], ru["y"], "b^", markersize=6)

# Plot UEs coloured by EVM
sc = ax.scatter(
    [u["x"] for u in ue_positions],
    [u["y"] for u in ue_positions],
    c=evm_arr, cmap="RdYlGn_r", s=80, edgecolors="black", linewidths=0.5, zorder=5,
)
for i, ue in enumerate(ue_positions):
    ax.annotate(str(i), (ue["x"], ue["y"]), fontsize=7, ha="center", va="bottom",
                xytext=(0, 5), textcoords="offset points")

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("EVM (%)")
margin = 1
ax.set_xlim(0 - margin, ROOM_X + margin)
ax.set_ylim(0 - margin, ROOM_Y + margin)
ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_title(f"{TEAM_NAME} — Coverage Map  |  ▲ = RU, ■ = CU, ● = UE")
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- CDF of EVM ------------------------------------------------------------
evm_sorted = np.sort(evm_arr)
cdf = np.arange(1, len(evm_sorted) + 1) / len(evm_sorted)

fig, ax = plt.subplots(figsize=(8, 4))
ax.step(evm_sorted, cdf, where="post", linewidth=2)
ax.axvline(mean_evm, color="blue", linestyle="--", label=f"Mean = {mean_evm:.2f}%")
ax.set_xlabel("EVM (%)")
ax.set_ylabel("CDF")
ax.set_title(f"{TEAM_NAME} — EVM CDF")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n Final Scores for {TEAM_NAME}:")
print(f"   Leaderboard 1 (quality):     {score_1:.4f}")
print(f"   Leaderboard 2 (cost-aware):  {score_2:.4f}")

In [ ]:
# Used to automatically push the results to a spreadsheet.
auth.authenticate_user()
creds, _ = default()

gc = gspread.authorize(creds)

worksheet = gc.open_by_key('1jG1zInePkmycIuZXTxpXwqGIE3aorGZ62i4PnUsmjL0').worksheet('Leaderboard')

# Read column A and find the team name
found_row = None
for row_index, cell in enumerate(worksheet.col_values(1), start=1):
    # Stop if we reach an empty cell
    if not cell or cell.strip() == '':
        break

    # Check if the cell value matches TEAM_NAME
    if cell.strip() == TEAM_NAME:
        found_row = row_index
        print(f"Found '{TEAM_NAME}' at row {found_row}")
        break

# Update the adjacent columns if team was found
if found_row is not None:
    # Update columns B and C in the found row with score_1 and score_2
    worksheet.update_cell(found_row, 2, score_1)  # Column B
    worksheet.update_cell(found_row, 3, score_2)  # Column C
    print(f"Updated row {found_row}: B{found_row}={score_1}, C{found_row}={score_2}")
else:
    print(f"Team '{TEAM_NAME}' not found in column A")